In [0]:
-- =====================================================================
-- SETUP
-- =====================================================================
CREATE OR REPLACE TEMP VIEW v_offer_windows AS
SELECT * FROM VALUES
    ('2024 Offer A', DATE'2024-11-29', DATE'2024-12-02'),
    ('2024 Offer B', DATE'2024-12-03', DATE'2024-12-31'),
    ('2025 Offer A', DATE'2025-05-09', DATE'2025-05-29'),
    ('2025 Offer B', DATE'2025-11-28', DATE'2025-12-01'),
    ('2025 Offer C', DATE'2025-12-11', DATE'2025-12-26'),
    ('2026 Offer A', DATE'2026-05-01', DATE'2026-05-31'),
    ('2026 Offer B', DATE'2026-06-15', DATE'2026-07-14')
AS w(window_name, start_date, end_date);

CREATE OR REPLACE TEMP VIEW v_events AS
SELECT cc_user_profile_id, event_type, plan_id, changed, changed_from, changed_to,
       CAST(reporting_datetime AS DATE)               AS rpt_date,
       CAST(from_unixtime(record_occurred_at) AS DATE) AS occurred_date
FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
WHERE cc_user_profile_id IS NOT NULL;

CREATE OR REPLACE TEMP VIEW v_last_date AS
SELECT MAX(rpt_date) AS last_date FROM v_events;

-- Every day from 2024-01-01 to latest data: year, Offer/Non-Offer, window
CREATE OR REPLACE TEMP VIEW v_periods AS
SELECT c.cal_date,
       YEAR(c.cal_date) AS yr,
       w.window_name,
       CASE WHEN w.window_name IS NOT NULL THEN 'Offer' ELSE 'Non-Offer' END AS period_type,
       COALESCE(w.window_name, CONCAT(YEAR(c.cal_date), ' Non-Offer'))      AS period_key
FROM (SELECT explode(sequence(DATE'2024-01-01', last_date, INTERVAL 1 DAY)) AS cal_date FROM v_last_date) c
LEFT JOIN v_offer_windows w ON c.cal_date BETWEEN w.start_date AND w.end_date;

-- Annual term starts: direct signup or upgrade took effect
CREATE OR REPLACE TEMP VIEW v_annual_starts AS
SELECT cc_user_profile_id, rpt_date AS start_date FROM v_events
WHERE event_type = 'subscription_created' AND plan_id = 'annual1' AND rpt_date IS NOT NULL
UNION
SELECT cc_user_profile_id, rpt_date FROM v_events
WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE AND rpt_date IS NOT NULL;

CREATE OR REPLACE TEMP VIEW v_annual_payments AS
SELECT DISTINCT cc_user_profile_id, rpt_date FROM v_events
WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1' AND rpt_date IS NOT NULL;

-- Renewal payment: annual1 payment >= 360 days after the latest annual start
CREATE OR REPLACE TEMP VIEW v_renewal_payments AS
SELECT ap.cc_user_profile_id, ap.rpt_date
FROM v_annual_payments ap
JOIN v_annual_starts s ON s.cc_user_profile_id = ap.cc_user_profile_id AND s.start_date <= ap.rpt_date
GROUP BY ap.cc_user_profile_id, ap.rpt_date
HAVING DATEDIFF(ap.rpt_date, MAX(s.start_date)) >= 360;

-- Every date an annual term begins (signup, upgrade took effect, or renewal)
CREATE OR REPLACE TEMP VIEW v_anchors AS
SELECT cc_user_profile_id, start_date AS anchor FROM v_annual_starts
UNION
SELECT cc_user_profile_id, rpt_date FROM v_renewal_payments;

-- Acquisitions: net new / returning signups + upgrades
-- Upgrade = Offer if its start OR took-effect date falls in an offer window
CREATE OR REPLACE TEMP VIEW v_acq_events AS
WITH first_created AS (
    SELECT cc_user_profile_id, MIN(rpt_date) AS first_created_date
    FROM v_events WHERE event_type = 'subscription_created'
    GROUP BY cc_user_profile_id
),
took_effect_by_period AS (   -- only used to drop upgrade completions logged as annual 'created'
    SELECT DISTINCT p.period_key, e.cc_user_profile_id
    FROM v_events e JOIN v_periods p ON e.rpt_date = p.cal_date
    WHERE e.event_type = 'subscription_cancelled' AND e.plan_id = 'monthly1' AND e.changed = TRUE
),
signups AS (
    SELECT p.yr, p.period_type, p.window_name, p.period_key, e.cc_user_profile_id, MIN(e.rpt_date) AS event_date
    FROM v_events e JOIN v_periods p ON e.rpt_date = p.cal_date
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
    GROUP BY p.yr, p.period_type, p.window_name, p.period_key, e.cc_user_profile_id
),
up_init AS (
    SELECT DISTINCT cc_user_profile_id, occurred_date AS init_date
    FROM v_events
    WHERE event_type = 'subscription_changed' AND changed = TRUE
      AND changed_from != changed_to AND changed_to = 'annual1' AND occurred_date IS NOT NULL
),
up_te AS (
    SELECT DISTINCT cc_user_profile_id, rpt_date AS te_date
    FROM v_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE AND rpt_date IS NOT NULL
),
paired AS (   -- each start matched to its took-effect within 35 days
    SELECT i.cc_user_profile_id, i.init_date, MIN(t.te_date) AS te_date
    FROM up_init i
    LEFT JOIN up_te t
      ON t.cc_user_profile_id = i.cc_user_profile_id
     AND t.te_date BETWEEN i.init_date AND DATE_ADD(i.init_date, 35)
    GROUP BY i.cc_user_profile_id, i.init_date
),
orphans AS (  -- took effect with no matching start
    SELECT t.cc_user_profile_id, CAST(NULL AS DATE) AS init_date, t.te_date
    FROM up_te t
    LEFT ANTI JOIN paired p ON p.cc_user_profile_id = t.cc_user_profile_id AND p.te_date = t.te_date
),
upgrades_credited AS (
    SELECT u.cc_user_profile_id, u.te_date,
           CASE WHEN pi.period_type = 'Offer' THEN u.init_date
                WHEN pt.period_type = 'Offer' THEN u.te_date
                ELSE COALESCE(u.init_date, u.te_date) END AS credit_date
    FROM (SELECT * FROM paired UNION ALL SELECT * FROM orphans) u
    LEFT JOIN v_periods pi ON pi.cal_date = u.init_date
    LEFT JOIN v_periods pt ON pt.cal_date = u.te_date
)
SELECT s.yr, s.period_type, s.window_name, s.period_key, s.cc_user_profile_id, s.event_date,
       CASE WHEN s.event_date = f.first_created_date THEN 'net_new' ELSE 'returning' END AS bucket,
       1 AS bucket_rank,
       s.event_date AS annual_start
FROM signups s
LEFT JOIN first_created f ON f.cc_user_profile_id = s.cc_user_profile_id
LEFT ANTI JOIN took_effect_by_period t ON t.period_key = s.period_key AND t.cc_user_profile_id = s.cc_user_profile_id
UNION ALL
SELECT p.yr, p.period_type, p.window_name, p.period_key, u.cc_user_profile_id, u.credit_date,
       'upgrade', 2,
       u.te_date      -- NULL if the upgrade never took effect
FROM upgrades_credited u
JOIN v_periods p ON p.cal_date = u.credit_date;


-- =====================================================================
-- TABLE 1 (Q1): 2025 acquisitions — Offer vs Non-Offer, with share %
-- =====================================================================
WITH d AS (
    SELECT * FROM (
        SELECT a.*, ROW_NUMBER() OVER (PARTITION BY a.period_type, a.cc_user_profile_id
                                       ORDER BY a.event_date, a.bucket_rank) AS rn
        FROM v_acq_events a WHERE a.yr = 2025
    ) WHERE rn = 1
),
agg AS (
    SELECT period_type,
           COUNT_IF(bucket = 'net_new')   AS net_new_signups,
           COUNT_IF(bucket = 'returning') AS returning_signups,
           COUNT_IF(bucket = 'upgrade')   AS upgrades,
           COUNT(*)                       AS total_acquired
    FROM d GROUP BY period_type
)
SELECT * FROM (
    SELECT period_type,
           net_new_signups,   ROUND(net_new_signups   / SUM(net_new_signups)   OVER (), 3) AS net_new_share,
           returning_signups, ROUND(returning_signups / SUM(returning_signups) OVER (), 3) AS returning_share,
           upgrades,          ROUND(upgrades          / SUM(upgrades)          OVER (), 3) AS upgrades_share,
           total_acquired,    ROUND(total_acquired    / SUM(total_acquired)    OVER (), 3) AS total_share
    FROM agg
    UNION ALL
    SELECT 'Total 2025',
           SUM(net_new_signups), 1.0, SUM(returning_signups), 1.0,
           SUM(upgrades), 1.0, SUM(total_acquired), 1.0
    FROM agg
)
ORDER BY CASE period_type WHEN 'Offer' THEN 0 WHEN 'Non-Offer' THEN 1 ELSE 2 END;


-- =====================================================================
-- TABLE 2 (Q2 + 2b): 2025 terms ended & renewal rate
--   overall, Offer vs Non-Offer (by term-end date), and per offer window
-- =====================================================================
WITH due AS (
    SELECT a.cc_user_profile_id, a.anchor, p.period_type, p.window_name
    FROM v_anchors a
    JOIN v_periods p ON p.cal_date = DATE_ADD(a.anchor, 365)
    WHERE p.yr = 2025
),
flag AS (
    SELECT d.cc_user_profile_id, d.period_type, d.window_name,
           MAX(CASE WHEN ap.cc_user_profile_id IS NOT NULL THEN 1 ELSE 0 END) AS renewed
    FROM due d
    LEFT JOIN v_annual_payments ap
      ON ap.cc_user_profile_id = d.cc_user_profile_id
     AND DATEDIFF(ap.rpt_date, d.anchor) BETWEEN 360 AND 385
    GROUP BY d.cc_user_profile_id, d.period_type, d.window_name
),
g AS (
    SELECT period_type, window_name,
           GROUPING(period_type) AS g_p, GROUPING(window_name) AS g_w,
           COUNT(DISTINCT cc_user_profile_id)                                AS terms_ended,
           COUNT(DISTINCT CASE WHEN renewed = 1 THEN cc_user_profile_id END) AS renewed
    FROM flag
    GROUP BY GROUPING SETS ((period_type, window_name), (period_type), ())
)
SELECT CASE WHEN g_p = 1 THEN 'Total 2025' ELSE period_type END AS term_ended_in,
       CASE WHEN g_w = 1 THEN 'ALL' ELSE window_name END       AS window_name,
       terms_ended, renewed,
       ROUND(renewed / NULLIF(terms_ended, 0), 3)              AS renewal_rate
FROM g
WHERE NOT (g_w = 0 AND window_name IS NULL)
ORDER BY g_p, CASE period_type WHEN 'Offer' THEN 0 ELSE 1 END, g_w DESC, window_name;


-- =====================================================================
-- TABLE 3 (2a): Renewal rate by acquisition cohort
--   every offer window + each year's Non-Offer, by bucket and ALL
-- =====================================================================
WITH d AS (
    SELECT * FROM (
        SELECT a.*, ROW_NUMBER() OVER (PARTITION BY a.period_key, a.cc_user_profile_id
                                       ORDER BY a.event_date, a.bucket_rank) AS rn
        FROM v_acq_events a
    ) WHERE rn = 1
),
ren AS (
    SELECT DISTINCT d.period_key, d.cc_user_profile_id
    FROM d
    JOIN v_annual_payments ap
      ON ap.cc_user_profile_id = d.cc_user_profile_id
     AND DATEDIFF(ap.rpt_date, d.annual_start) BETWEEN 360 AND 385
)
SELECT d.yr, d.period_type, d.period_key AS acquisition_cohort,
       COALESCE(d.bucket, 'ALL') AS bucket,
       COUNT(DISTINCT d.cc_user_profile_id)                                           AS acquired,
       COUNT(DISTINCT CASE WHEN d.annual_start IS NULL THEN d.cc_user_profile_id END) AS upgrade_not_completed,
       COUNT(DISTINCT CASE WHEN DATE_ADD(d.annual_start, 385) >  l.last_date THEN d.cc_user_profile_id END) AS not_yet_eligible,
       COUNT(DISTINCT CASE WHEN DATE_ADD(d.annual_start, 385) <= l.last_date THEN d.cc_user_profile_id END) AS eligible,
       COUNT(DISTINCT CASE WHEN DATE_ADD(d.annual_start, 385) <= l.last_date
                            AND r.cc_user_profile_id IS NOT NULL THEN d.cc_user_profile_id END)           AS renewed,
       ROUND(
         COUNT(DISTINCT CASE WHEN DATE_ADD(d.annual_start, 385) <= l.last_date
                              AND r.cc_user_profile_id IS NOT NULL THEN d.cc_user_profile_id END)
         / NULLIF(COUNT(DISTINCT CASE WHEN DATE_ADD(d.annual_start, 385) <= l.last_date
                                      THEN d.cc_user_profile_id END), 0), 3)                              AS renewal_rate
FROM d
CROSS JOIN v_last_date l
LEFT JOIN ren r ON r.period_key = d.period_key AND r.cc_user_profile_id = d.cc_user_profile_id
GROUP BY GROUPING SETS ((d.yr, d.period_type, d.period_key, d.bucket), (d.yr, d.period_type, d.period_key))
ORDER BY d.yr, CASE d.period_type WHEN 'Offer' THEN 0 ELSE 1 END, acquisition_cohort, d.bucket IS NULL, d.bucket;